# MedShapeNet: Kaggle Training Notebook (2D → 3D)

Этот ноутбук показывает **какой датасет нужен** и как запустить минимальное обучение на Kaggle для синтеза 3D-моделей из 2D изображений.

Рекомендация для старта на Kaggle:
- 1 категория (например `liver` или `vertebrae`)
- 200–1000 STL объектов
- `voxel_size=32` или `64`
- 3 проекции (`axial`, `sagittal`, `coronal`)


## 1) Environment
Добавьте репозиторий как Kaggle Dataset или загрузите его в `/kaggle/working/MedShapeNetDataset`.


In [ ]:
import os
import sys
from pathlib import Path

CANDIDATE_REPO_ROOTS = [
    Path('/kaggle/input/medshapenetdataset'),
    Path('/kaggle/input/medshapenet-dataset'),
    Path('/kaggle/working/MedShapeNetDataset'),
]

REPO_ROOT = None
for p in CANDIDATE_REPO_ROOTS:
    if (p / 'MedShapeNetDataset.txt').exists() and (p / 'src').exists():
        REPO_ROOT = p
        break

if REPO_ROOT is None:
    raise FileNotFoundError('Repo not found. Put repository into /kaggle/input/... or /kaggle/working/MedShapeNetDataset')

print('Using REPO_ROOT =', REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

WORK_DIR = Path('/kaggle/working/medshapenet_run')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_DIR =', WORK_DIR)


In [ ]:
# Install deps in Kaggle if needed
# Uncomment if your environment does not already have them:
# !pip install -q -r /kaggle/input/medshapenetdataset/requirements.txt


## 2) Show dataset categories and pick subset


In [ ]:
from collections import Counter

url_file = REPO_ROOT / 'MedShapeNetDataset.txt'
counter = Counter()
with open(url_file) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        name = Path(line).name
        stem = name[:-4] if name.endswith('.stl') else name
        parts = stem.split('_', 1)
        if len(parts) == 2:
            counter[parts[1]] += 1

print('Unique categories:', len(counter))
for cat, n in counter.most_common(20):
    print(f'{cat:35s} {n}')


## 3) Download training subset
Рекомендуется начать с одной категории, например `liver`/`vertebrae`/`tumoredbrain`.


In [ ]:
from src.medshapenet.dataset import MedShapeNetDownloader

TARGET_CATEGORY = 'liver'
MAX_SAMPLES = 300  # для Kaggle demo
STL_DIR = WORK_DIR / 'stl'

downloader = MedShapeNetDownloader(url_file)
downloaded = downloader.download(
    category=TARGET_CATEGORY,
    max_samples=MAX_SAMPLES,
    output_dir=STL_DIR,
    skip_existing=True,
)
print('Downloaded files:', len(downloaded))


## 4) Preprocess STL → voxels + 2D views


In [ ]:
from src.medshapenet.preprocess import batch_preprocess

PROCESSED_DIR = WORK_DIR / 'processed'
batch_preprocess(
    stl_dir=STL_DIR,
    output_dir=PROCESSED_DIR,
    voxel_size=32,
    image_size=128,
    num_views=3,
    overwrite=False,
)
print('Preprocessing done:', PROCESSED_DIR)


## 5) Train (light Kaggle config)


In [ ]:
import yaml
from src.medshapenet.train import train

cfg_path = REPO_ROOT / 'configs' / 'default.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['data_dir'] = str(PROCESSED_DIR)
cfg['dataset']['category'] = TARGET_CATEGORY
cfg['dataset']['voxel_size'] = 32
cfg['dataset']['num_views'] = 3

cfg['training']['batch_size'] = 4
cfg['training']['num_epochs'] = 3
cfg['training']['num_workers'] = 2
cfg['training']['learning_rate'] = 1e-4

cfg['checkpointing']['checkpoint_dir'] = str(WORK_DIR / 'checkpoints')
cfg['checkpointing']['save_every_n_epochs'] = 1

train(cfg)


## 6) Inference + STL export


In [ ]:
from src.medshapenet.inference import Reconstructor\n\nbest_ckpt = WORK_DIR / 'checkpoints' / 'best.pth'\nsample_dirs = sorted((PROCESSED_DIR / TARGET_CATEGORY).glob('*/'))\nif not sample_dirs:\n    raise RuntimeError('No preprocessed samples found. Check download/preprocess steps.')\nsample_dir = sample_dirs[0]\nviews = sorted(sample_dir.glob('view_*.npy'))[:3]\nif len(views) < 1:\n    raise RuntimeError(f'No view_*.npy files in {sample_dir}')\n\nrec = Reconstructor(best_ckpt)\nvoxels = rec.reconstruct_from_files(views)\nout_stl = WORK_DIR / f'{TARGET_CATEGORY}_kaggle_result.stl'\nrec.save_stl(voxels, out_stl)\nprint('Saved:', out_stl)\n